In [1]:
from pathlib import Path
import json
import pandas as pd
import pickle
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import textwrap

In [2]:
class Colors:
    red = '#da291c'
    medium_red = '#ed7b73'
    light_red = '#f9d3d0'
    purple = '#ae90c3'
    medium_purple = '#cebcdb'
    light_purple = '#efe9f3'
    yellow = '#fecf28'
    medium_yellow = '#ffeca9'
    light_yellow = '#fff5d4'
    blue = '#00B2A9'
    medium_blue = '#66CCC6'
    light_blue = '#CCEEEC'
    green = '#45b283'
    medium_green = '#8dd3b5'
    light_green = '#d9f0e6'
    grey_1000 = '#1F1F1F'
    grey_900 = '#2B2B2B'
    grey_700 = '#666666'
    grey_500 = '#9E9E9E'
    grey_300 = '#D9D9D9'
    grey_200 = '#EEEEEE'
    grey_100 = '#F5F5F5'


ordinal_3 = [
    Colors.red,
    Colors.grey_300,
    Colors.green
]

ordinal_4 = [
    Colors.red,
    Colors.medium_red,
    Colors.medium_green,
    Colors.green
]

ordinal_5 = [
    Colors.red,
    Colors.medium_red,
    Colors.grey_300,
    Colors.medium_green,
    Colors.green
]

ordinal_6 = [
    Colors.red,
    Colors.medium_red,
    Colors.light_red,
    Colors.light_green,
    Colors.medium_green,
    Colors.green
]

ordinal_7 = [
    Colors.red,
    Colors.medium_red,
    Colors.light_red,
    Colors.grey_300,
    Colors.light_green,
    Colors.medium_green,
    Colors.green
]

def get_ordinal_colors(n):
    palettes = {
        3: ordinal_3,
        4: ordinal_4,
        5: ordinal_5,
        6: ordinal_6,
        7: ordinal_7
    }
    return palettes.get(n)

def sc_theme():

    mpl.rcParams['font.family'] = 'sans-serif'
    mpl.rcParams['font.sans-serif'] = ['Segoe UI', 'Arial']

    mpl.rcParams.update({
        "figure.facecolor": Colors.grey_100,
        "axes.facecolor": Colors.grey_200,
        "axes.edgecolor": Colors.grey_500,
        "axes.labelcolor": Colors.grey_1000,
        "xtick.color": Colors.grey_1000,
        "ytick.color": Colors.grey_1000,
        "text.color": Colors.grey_1000,
        "axes.spines.top": False,
        "axes.spines.right": False,
        'ytick.major.size': 0,
        'xtick.major.size': 0,
        'xtick.minor.size': 0,
        'ytick.minor.size': 0,
        'axes.grid': True,
        'grid.color': Colors.grey_100,
        'grid.alpha': 1.0,
        'grid.linestyle': '-',
        'grid.linewidth': 1.5,
        'savefig.bbox': 'tight',
        'savefig.dpi': 300,
        'savefig.pad_inches': 0.3,
        'axes.titlepad': 12
    })

In [3]:
class DataSet:
    def __init__(self, df, qdf, cdf, metadata):
        self.df = df
        self.qdf = qdf
        self.cdf = cdf
        self.metadata = metadata

In [4]:
def find_project_root(start_path: Path, marker_file: str = 'path_config.json') -> Path:

    for candidate in [start_path.resolve(), *start_path.resolve().parents]:
        if (candidate / marker_file).exists():
            return candidate
    raise FileNotFoundError(f'Could not find {marker_file} above {start_path}')
        
PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "path_config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PATHS = json.load(f)

CONFIG_DIR = (PROJECT_ROOT / PATHS["config_dir"]).resolve()

with open(CONFIG_DIR / "dimensions_config.json", "r", encoding="utf-8") as f:
            dimensions_config = json.load(f)

with open(CONFIG_DIR / "child_config.json", "r", encoding="utf-8") as f:
            child_config = json.load(f)

In [5]:
def classify_series(col):
    """Classify a series and return its category plus a simple range descriptor."""
    nunique = col.nunique(dropna=True)

    if pd.api.types.is_numeric_dtype(col):
        if nunique > 10:
            return "numeric", (col.min(), col.max())
        return "ordinal", nunique

    return "categorical", nunique


def data_map(dataset):
    dimensions = dataset.df_summary["available_dimensions"]
    indicators = dataset.df_summary["available_indicators"]

    rows = []

    for var_type, names in [("indicator", indicators), ("dimension", dimensions)]:
        for name in names:
            col = dataset.df_short[name]
            category, value_range = classify_series(col)
            rows.append(
                {
                    "name": name,
                    "ind_type": var_type,
                    "category": category,
                    "range": value_range,
                }
            )

    return pd.DataFrame(rows)

In [6]:


def plotter(kind, plot, df, dim, ind):

    ind_norm = ind.replace("_", " ").capitalize()
    dim_norm = dim.replace("_", " ").capitalize()    
    
    if kind == "static":
        fig, ax = plt.subplots(figsize=(10, 6))

        if plot == "mean_bar":
            sns.barplot(x=dim, y=ind, data=df, color=Colors.medium_green, errorbar=None, ax=ax)
            ax.grid(False)

        elif plot == "stacked_bar":
            ct = pd.crosstab(df[dim], df[ind], normalize="index")
            n = ct.shape[1]
            colors = get_ordinal_colors(n)
            ct.plot(kind="bar", stacked=True, color=colors, ax=ax)
            ax.grid(False)
        
        elif plot == "scatter":
            sns.scatterplot(x=dim, y=ind, data=df, color=Colors.medium_green, ax=ax)
        
        else:
            raise ValueError(f"Plot type {plot} not recognized.")

        ax.set_title(f"Plot of {ind_norm} by {dim_norm}", fontsize=16, fontname="Oswald")
        ax.set_xticks(ax.get_xticks())
        ax.set_xticklabels([(textwrap.fill(str(label.get_text()), 20)).replace("_", " ").capitalize() for label in ax.get_xticklabels()], rotation=0, fontsize=10)
        ax.set_xlabel(dim_norm, fontsize=12)
        ax.set_ylabel(ind_norm, fontsize=12)
        ax.legend(loc="upper left", bbox_to_anchor=(1, 1), frameon=False)

        return fig

    if kind == "dynamic":
        if plot == "mean_bar":

            df_mean = df.groupby(dim)[ind].mean().reset_index()
            fig = px.bar(
                df_mean,
                x=dim,
                y=ind,
                color_discrete_sequence=[Colors.medium_green]
            )

            fig.update_traces(hovertemplate=f"{dim_norm}: %{{x}}<br>Mean {ind_norm}: %{{y:.2f}}<extra></extra>")

        elif plot == "stacked_bar":
            ct = pd.crosstab(df[dim], df[ind], normalize="index")
            ct = ct.sort_index(axis=1)

            n = ct.shape[1]
            colors = get_ordinal_colors(n)

            fig = px.bar(
                ct,
                x=ct.index,
                y=ct.columns,
                color_discrete_sequence=colors
            )

            fig.update_traces(hovertemplate=f"{dim_norm}: %{{x}}<br>{ind_norm}: %{{fullData.name}}<br>%{{y:.1%}}<extra></extra>")

        elif plot == "scatter":
            fig = px.scatter(
                df,
                x=dim,
                y=ind,
                color_discrete_sequence=[Colors.medium_green]
            )
            
            fig.update_traces(
                marker=dict(color=Colors.medium_green),
                hovertemplate=f"{dim_norm}: %{{x}}<br>{ind_norm}: %{{y:.2f}}<extra></extra>"
            )

            fig.update_layout(hovermode="closest")

            fig.update_xaxes(showspikes=True, spikecolor=Colors.grey_500)
            fig.update_yaxes(showspikes=True, spikecolor=Colors.grey_500)


        else:
            raise ValueError(f"Plot type {plot} not recognized.")

        # ---- Styling / layout ----
        fig.update_layout(
            title=f"Plot of {ind_norm} by {dim_norm}",
            xaxis_title=dim_norm,
            yaxis_title=ind_norm,
            template="simple_white",
            width=800,
            height=500,
            showlegend=True
        )

        # Wrap x-axis labels (approximation)
        fig.update_xaxes(
            ticktext=[textwrap.fill(str(x), 10) for x in df[dim].unique()],
            tickvals=df[dim].unique()
        )

        return fig

In [7]:
def fig_list(dataset, preferred_plot=None, max_figs=10):
    """Build and filter the final figure specification list."""
    if not hasattr(dataset, "data_map") or dataset.data_map is None:
        dataset.data_map = data_map(dataset)

    ind_df = dataset.data_map[dataset.data_map["ind_type"] == "indicator"]
    dim_df = dataset.data_map[dataset.data_map["ind_type"] == "dimension"]

    pairs = ind_df.merge(dim_df, how="cross", suffixes=("_ind", "_dim"))

    def get_plots(ind_cat, dim_cat):
        plots = []

        if dim_cat in ["categorical", "ordinal"]:
            if ind_cat in ["numeric", "ordinal"]:
                plots.append("mean_bar")
            if ind_cat == "ordinal":
                plots.append("stacked_bar")

        if dim_cat == "numeric" and ind_cat in ["numeric", "ordinal"]:
            plots.append("scatter")

        return plots

    rows = []
    for _, r in pairs.iterrows():
        plots = get_plots(r["category_ind"], r["category_dim"])
        for p in plots:
            rows.append(
                {
                    "indicator": r["name_ind"],
                    "dimension": r["name_dim"],
                    "plot": p,
                }
            )

    fig_map = pd.DataFrame(rows)

    if fig_map.empty:
        return fig_map

    if preferred_plot:
        reduced_rows = []
        for (_, _), group in fig_map.groupby(["indicator", "dimension"]):
            if preferred_plot in group["plot"].values:
                reduced_rows.append(group[group["plot"] == preferred_plot].iloc[0])
            else:
                reduced_rows.append(group.iloc[0])
        fig_map = pd.DataFrame(reduced_rows)
    else:
        fig_map = fig_map.drop_duplicates(["indicator", "dimension"], keep="first")

    return fig_map.head(max_figs).reset_index(drop=True)


def show_figures(dataset, kind="plotly", preferred_plot=None, max_figs=10):
    specs = fig_list(dataset, preferred_plot=preferred_plot, max_figs=max_figs)

    if specs.empty:
        print("No figures to plot.")
        return []

    figs = []
    for row in specs.itertuples(index=False):
        ind = row.indicator
        dim = row.dimension
        plot = row.plot

        df = dataset.df_short[[ind, dim]].dropna()
        fig = plotter(kind, plot, df, dim, ind)
        figs.append(fig)

    return figs, specs

In [32]:
with open("../data/datasets/aser_test.pkl", "rb") as f:
    dataset = pickle.load(f)

In [10]:
figs, specs = show_figures(dataset, kind="dynamic", preferred_plot="stacked_bar")

In [ ]:
specs

,indicator,dimension,plot
0,aser_literacy,admin1,stacked_bar
1,aser_literacy,age,stacked_bar
2,aser_literacy,age_group,stacked_bar
3,aser_literacy,disability,stacked_bar
4,aser_literacy,location,stacked_bar
5,aser_literacy,protection_status,stacked_bar
6,aser_literacy,school,stacked_bar
7,aser_literacy,sex,stacked_bar
8,aser_numeracy,admin1,stacked_bar
9,aser_numeracy,age,stacked_bar


In [ ]:
figs[8]